# HPRA Medicines Scraper — 3-Phase Pipeline  *(Phase 3 fixed v4)*

| Phase | What it does | Output |
|---|---|---|
| **1** | API → all product info + leaflet URLs | `hpra_phase1.csv` |
| **2** | Download every Package Leaflet PDF | `pdfs/{id}.pdf` |
| **3** | Extract Manufacturer / Address / Country from PDFs | `hpra_final.csv` |

> **v4 fixes** — multi-manufacturer wrapped blocks now fully extracted; lowercase company names (betapharm, etc.) now recognised; country detection limited to last address part only (prevents false city matches); postal code prefixes DK-/SE-/CH-/A-/B-/NL- detected; expanded Romanian/Portuguese/UK city coverage.

**Each phase checks if it's already done and skips automatically.**  
Run **Cell 1** (setup) then **Cell 2** (status check) — it tells you exactly what will run.


## Cell 1 — Setup (run once)

In [88]:
# ── Install ───────────────────────────────────────────────────────────────────
%pip install -q requests beautifulsoup4 lxml pdfplumber pandas openpyxl tqdm playwright
import subprocess, sys
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium", "--with-deps"],
               check=True, capture_output=True)

# ── Imports ───────────────────────────────────────────────────────────────────
import csv, json, logging, os, re, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pdfplumber
import pandas as pd
from tqdm.auto import tqdm
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ── Config ────────────────────────────────────────────────────────────────────
PHASE1_CSV    = "hpra_phase1.csv"
PDF_DIR       = Path("pdfs")
FINAL_CSV     = "hpra_final.csv"
OUTPUT_XLSX   = "hpra_medicines.xlsx"

WORKERS       = 5
REQUEST_DELAY = 1.0
TIMEOUT       = 30
BATCH_SIZE    = 100

BASE_URL      = "https://www.hpra.ie"
DETAIL_URL    = BASE_URL + "/find-a-medicine/for-human-use/authorised-medicines/details/{id}"
MEDICINES_URL = BASE_URL + "/find-a-medicine/for-human-use/authorised-medicines"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-IE,en;q=0.9",
}

PHASE1_FIELDS = ["product_id","product_name","licence_number","licence_holder",
                 "active_substances","atc_code","package_leaflet_url","error"]
FINAL_FIELDS  = ["product_id","product_name","licence_number","licence_holder",
                 "active_substances","atc_code","package_leaflet_url",
                 "mfr_company","mfr_address","mfr_country","error"]

PDF_DIR.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler("hpra.log", encoding="utf-8")],
)
log = logging.getLogger(__name__)
logging.getLogger("pdfminer").setLevel(logging.ERROR)  # suppress noisy color-space warnings

session = requests.Session()
session.headers.update(HEADERS)

# ── HTTP helpers ──────────────────────────────────────────────────────────────
def _get(url):
    for attempt in range(3):
        try:
            time.sleep(REQUEST_DELAY)
            r = session.get(url, timeout=TIMEOUT)
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            log.warning("GET %d/3: %s", attempt+1, exc)
            if attempt == 2: raise
            time.sleep(5*(attempt+1))

def _post(url, body, extra_headers=None):
    hdrs = {**HEADERS, "Content-Type": "application/json"}
    if extra_headers: hdrs.update(extra_headers)
    for attempt in range(3):
        try:
            time.sleep(REQUEST_DELAY)
            r = session.post(url, json=body, headers=hdrs, timeout=TIMEOUT)
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            log.warning("POST %d/3: %s", attempt+1, exc)
            if attempt == 2: raise
            time.sleep(5*(attempt+1))

def load_done(csv_path, id_col="product_id"):
    if os.path.exists(csv_path):
        return set(pd.read_csv(csv_path, usecols=[id_col])[id_col].astype(str))
    return set()

def append_rows(rows, csv_path, fields):
    first = not os.path.exists(csv_path)
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        if first: w.writeheader()
        w.writerows(rows)

print("✓ Setup complete")



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✓ Setup complete


## Cell 2 — Status check

Run this to see what's already done and what will run next.

In [90]:
# ── Check Phase 1 ─────────────────────────────────────────────────────────────
phase1_rows   = len(pd.read_csv(PHASE1_CSV)) if os.path.exists(PHASE1_CSV) else 0
phase1_done   = phase1_rows > 0

# ── Check Phase 2 ─────────────────────────────────────────────────────────────
pdfs_on_disk  = len(list(PDF_DIR.glob("*.pdf")))

if phase1_done:
    df1       = pd.read_csv(PHASE1_CSV)
    urls_total = df1["package_leaflet_url"].notna() & df1["package_leaflet_url"].astype(str).str.startswith("http")
    pdf_total  = urls_total.sum()
    pdf_remaining = pdf_total - pdfs_on_disk
else:
    pdf_total, pdf_remaining = "?", "?"

phase2_done = isinstance(pdf_remaining, int) and pdf_remaining <= 0

# ── Check Phase 3 ─────────────────────────────────────────────────────────────
final_rows    = len(pd.read_csv(FINAL_CSV)) if os.path.exists(FINAL_CSV) else 0
phase3_done   = final_rows > 0 and final_rows >= pdfs_on_disk

# ── Print summary ──────────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════╗")
print("║              HPRA Scraper — Status                  ║")
print("╠══════════════════════════════════════════════════════╣")

p1_status = "✅ DONE" if phase1_done   else "⏳ TODO"
p2_status = "✅ DONE" if phase2_done   else "⏳ TODO"
p3_status = "✅ DONE" if phase3_done   else "⏳ TODO"

print(f"║  Phase 1 — Product info      {p1_status}               ║")
print(f"║    {phase1_rows:,} rows in {PHASE1_CSV:<25}     ║")
print(f"║                                                      ║")
print(f"║  Phase 2 — Download PDFs     {p2_status}               ║")
print(f"║    {pdfs_on_disk:,} PDFs on disk / {pdf_total} with URL          ║")
print(f"║                                                      ║")
print(f"║  Phase 3 — Extract from PDFs {p3_status}               ║")
print(f"║    {final_rows:,} rows in {FINAL_CSV:<25}      ║")
print("╚══════════════════════════════════════════════════════╝")

if not phase1_done:
    print("\n▶ Next: run Cell 3 (Phase 1 — collect product info)")
elif not phase2_done:
    print(f"\n▶ Next: run Cell 4 (Phase 2 — download {pdf_remaining:,} remaining PDFs)")
elif not phase3_done:
    remaining3 = pdfs_on_disk - final_rows
    print(f"\n▶ Next: run Cell 5 (Phase 3 — extract from {remaining3:,} remaining PDFs)")
else:
    print("\n▶ All phases done! Run Cell 6 to export the final Excel file.")


╔══════════════════════════════════════════════════════╗
║              HPRA Scraper — Status                  ║
╠══════════════════════════════════════════════════════╣
║  Phase 1 — Product info      ✅ DONE               ║
║    10,305 rows in hpra_phase1.csv               ║
║                                                      ║
║  Phase 2 — Download PDFs     ⏳ TODO               ║
║    6,296 PDFs on disk / 6297 with URL          ║
║                                                      ║
║  Phase 3 — Extract from PDFs ✅ DONE               ║
║    6,296 rows in hpra_final.csv                 ║
╚══════════════════════════════════════════════════════╝

▶ Next: run Cell 4 (Phase 2 — download 1 remaining PDFs)


## Cell 3 — Phase 1: Collect product info & leaflet URLs

In [92]:
# ── Skip check ────────────────────────────────────────────────────────────────
phase1_rows = len(pd.read_csv(PHASE1_CSV)) if os.path.exists(PHASE1_CSV) else 0
if phase1_rows > 0:
    print(f"✅ Phase 1 already done — {phase1_rows:,} rows in {PHASE1_CSV}")
    print("   Loading existing data…")
    df1 = pd.read_csv(PHASE1_CSV)
    all_stubs = df1[["product_id","product_name","licence_number","licence_holder"]].to_dict("records")
    all_stubs = [{k: str(v) for k, v in r.items()} for r in all_stubs]
    print(f"   {len(all_stubs):,} stubs loaded. Skipping to next phase.")
else:
    # ── 1a: Discover API via Playwright ───────────────────────────────────────
    from playwright.async_api import async_playwright
    _api_url, _api_body, _api_hdrs = None, {}, {}

    async def discover_api():
        global _api_url, _api_body, _api_hdrs
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            ctx  = await browser.new_context(user_agent=HEADERS["User-Agent"], locale="en-IE")
            page = await ctx.new_page()
            async def on_req(req):
                global _api_url, _api_body, _api_hdrs
                if "HumanApprovedProducts" in req.url or ("sfapi" in req.url and req.method=="POST"):
                    _api_url  = req.url
                    _api_hdrs = dict(req.headers)
                    try:    _api_body = json.loads(req.post_data or "{}")
                    except: _api_body = {}
            page.on("request", on_req)
            print("Launching headless Chrome to discover API…")
            await page.goto(MEDICINES_URL, wait_until="networkidle", timeout=60_000)
            await page.wait_for_timeout(5_000)
            await browser.close()

    await discover_api()
    assert _api_url, "API not found — set _api_url manually"
    print(f"✅ API: {_api_url[:80]}…")

    # ── 1b: Fetch all product stubs from API ──────────────────────────────────
    BASE_BODY = {"skip":0,"take":10,"query":None,"order":"LastUpdated DESC","status":"Authorised",
                 "filters":{"from":None,"to":None,"routes":None,"placeOfSale":None,
                             "advertisingConditions":None,"marketingAvailability":None,
                             "atcCode":None,"medicineType":None}}
    winning_body = None
    for body in [_api_body, BASE_BODY, {"skip":0,"take":5,"status":"Authorised"}, {}]:
        try:
            r = _post(_api_url, body, _api_hdrs)
            if r.status_code != 200: continue
            data  = r.json()
            items = data if isinstance(data,list) else next(
                (v for v in (data.values() if isinstance(data,dict) else []) if isinstance(v,list)), None)
            if items: winning_body = body; break
        except: pass
    assert winning_body is not None, "No working API body"

    def _items_total(data):
        if isinstance(data,list): return data,None
        if isinstance(data,dict):
            for k in ("results","items","products","medicines","data","value","Records","Products"):
                if k in data and isinstance(data[k],list):
                    t=next((int(data[x]) for x in ("total","totalCount","Total","TotalRecords","count") if x in data),None)
                    return data[k],t
            for v in data.values():
                if isinstance(v,list): return v,None
        return [],None

    def _to_stub(item):
        pid=str(item.get("id") or item.get("Id") or item.get("productId") or item.get("ProductId") or "")
        return {"product_id":pid,
                "product_name": item.get("productName") or item.get("ProductName") or "",
                "licence_number": item.get("licenceNumber") or item.get("LicenceNumber") or "",
                "licence_holder": item.get("licenceHolder") or item.get("LicenceHolder") or item.get("company") or ""}

    stubs, skip, total = [], 0, None
    while True:
        body = {**winning_body,"skip":skip,"take":100}
        items,t = _items_total(_post(_api_url, body, _api_hdrs).json())
        if t is not None and total is None: total=t; log.info("API total: %d",total)
        if not items: break
        stubs.extend(_to_stub(i) for i in items)
        log.info("skip=%d got=%d running=%d",skip,len(items),len(stubs))
        if (total and len(stubs)>=total) or len(items)<100: break
        skip+=100
    seen,all_stubs=[],[]
    for s in stubs:
        if s["product_id"] not in seen: seen.append(s["product_id"]); all_stubs.append(s)
    print(f"✅ {len(all_stubs):,} products from API")

    # ── 1c: Scrape detail pages ────────────────────────────────────────────────
    def scrape_detail(stub):
        result={**stub,"active_substances":"","atc_code":"","package_leaflet_url":"","error":""}
        try:
            r=_get(DETAIL_URL.format(id=stub["product_id"]))
            soup=BeautifulSoup(r.text,"html.parser")
            dl={}
            for dt in soup.find_all("dt"):
                lbl=dt.get_text(strip=True).lower().rstrip(":")
                dd=dt.find_next_sibling("dd")
                if dd: dl[lbl]=dd.get_text(" ",strip=True)
            for k in ("active substances","active substance"):
                if k in dl: result["active_substances"]=dl[k]; break
            for k in ("atc code","atc"):
                if k in dl: result["atc_code"]=dl[k]; break
            for a in soup.find_all("a",href=True):
                if "package leaflet" in a.get_text(strip=True).lower():
                    h=a["href"]
                    result["package_leaflet_url"]=h if h.startswith("http") else BASE_URL+h
                    break
            for k in ("licence number","license number"):
                if k in dl and not result["licence_number"]: result["licence_number"]=dl[k]; break
            for k in ("licence holder","license holder","marketing authorisation holder"):
                if k in dl and not result["licence_holder"]: result["licence_holder"]=dl[k]; break
        except Exception as exc:
            result["error"]=str(exc)
        return result

    done_ids  = load_done(PHASE1_CSV)
    remaining = [s for s in all_stubs if s["product_id"] not in done_ids]
    print(f"Detail pages: {len(done_ids):,} done, {len(remaining):,} remaining")

    batch=[]
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futures={ex.submit(scrape_detail,s):s for s in remaining}
        pbar=tqdm(as_completed(futures),total=len(remaining),desc="Phase 1",unit="product")
        for f in pbar:
            row=f.result(); batch.append(row)
            if len(batch)>=BATCH_SIZE: append_rows(batch,PHASE1_CSV,PHASE1_FIELDS); batch.clear()
    if batch: append_rows(batch,PHASE1_CSV,PHASE1_FIELDS)

    df1=pd.read_csv(PHASE1_CSV)
    print(f"\n✅ Phase 1 complete — {len(df1):,} rows → {PHASE1_CSV}")


✅ Phase 1 already done — 10,305 rows in hpra_phase1.csv
   Loading existing data…
   10,305 stubs loaded. Skipping to next phase.


## Cell 4 — Phase 2: Download PDFs

In [94]:
# ── Skip check ────────────────────────────────────────────────────────────────
df1       = pd.read_csv(PHASE1_CSV)
has_url   = df1[df1["package_leaflet_url"].notna() &
                df1["package_leaflet_url"].astype(str).str.startswith("http")]
already   = {p.stem for p in PDF_DIR.glob("*.pdf")}
to_dl     = has_url[~has_url["product_id"].astype(str).isin(already)]

print(f"Products with leaflet URL : {len(has_url):,}")
print(f"PDFs already on disk      : {len(already):,}")
print(f"To download               : {len(to_dl):,}")

if len(to_dl) == 0:
    print("\n✅ Phase 2 already done — all PDFs downloaded. Skipping.")
else:
    def download_pdf(row):
        pid  = str(row["product_id"])
        path = PDF_DIR / f"{pid}.pdf"
        try:
            r=_get(row["package_leaflet_url"])
            path.write_bytes(r.content)
            return pid,True,""
        except Exception as exc:
            return pid,False,str(exc)

    ok,fail=0,0
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futures={ex.submit(download_pdf,row):row for _,row in to_dl.iterrows()}
        pbar=tqdm(as_completed(futures),total=len(to_dl),desc="Phase 2 — PDFs",unit="pdf")
        for f in pbar:
            pid,success,err=f.result()
            if success: ok+=1
            else: fail+=1
            pbar.set_postfix(ok=ok,fail=fail)

    print(f"\n✅ Phase 2 complete — downloaded {ok:,}, failed {fail:,}")
    print(f"   Total PDFs on disk: {len(list(PDF_DIR.glob('*.pdf'))):,}")


Products with leaflet URL : 6,297
PDFs already on disk      : 6,296
To download               : 1


Phase 2 — PDFs:   0%|          | 0/1 [00:00<?, ?pdf/s]

2026-08-22 16:09:18,582  WARNING   GET 1/3: 404 Client Error: The specified blob does not exist. for url: https://assets.hpra.ie/products/BlobNotFound
2026-08-22 16:09:24,742  WARNING   GET 2/3: 404 Client Error: The specified blob does not exist. for url: https://assets.hpra.ie/products/BlobNotFound
2026-08-22 16:09:35,923  WARNING   GET 3/3: 404 Client Error: The specified blob does not exist. for url: https://assets.hpra.ie/products/BlobNotFound

✅ Phase 2 complete — downloaded 0, failed 1
   Total PDFs on disk: 6,296


## Cell 5 — Phase 3: Extract Manufacturer / Address / Country from PDFs

In [96]:
## Cell 5 — Phase 3: Extract Manufacturer / Address / Country from PDFs
#
# FIXES vs. original (6 bugs + 2 improvements):
#
#  BUG 1 (primary, ~89% failure): re.searatch typo → AttributeError on every
#         non-Manufacturer line → silent empty result for almost all PDFs.
#         Fix: removed the redundant+broken re.searatch; keep only _MFR_ONLY.match().
#
#  BUG 2: regex r"manufacturer" doesn't match "Manufactured by:" (ends in 'd'
#         not 'r') → entire page skipped for leaflets that use "Manufactured by:".
#         Fix: changed to r"manufactur" (stem matches both forms).
#
#  BUG 3: parse_inline_record searched all parts including parts[0] (company name)
#         for a country match → "Meiji Pharma Spain, S.A." falsely matched "Spain"
#         as the country, stripping it from the company name.
#         Fix: country search now starts at parts[1] (never parts[0]).
#
#  BUG 4: extract_all_from_block broke on any failed parse even when the failed
#         line was an address continuation (e.g. "de Henares (Guadalajara), Spain")
#         → stopped before collecting subsequent manufacturers (Losartan had 3).
#         Fix: lines starting with lowercase/digit/dash are treated as continuations
#         and skipped; only uppercase-initial unparseable lines break the loop.
#
#  BUG 5: column-detection spanning check used per-row aggregation (all words at
#         the same y merged into one "row") → two-column pages where both columns
#         share y-levels (e.g. Topamax) incorrectly classified as single-column.
#         Fix: count individual words that physically cross the split, not rows.
#
#  IMPROVEMENT A: _extract_page_text now detects and handles 3-column layouts.
#  IMPROVEMENT B: _collect_block / _SKIP_LINE stop on "This medicinal product..."
#                 preventing EEA member-state table rows leaking into the address.
#
#  BUG 7: _extract_page_text's column-layout detector produced FALSE
#         POSITIVES on ordinary single-column pages -- a coincidental gap
#         in word start-positions (e.g. a short footer/page-number line)
#         got misread as a column boundary, and the page was sliced in
#         half mid-word/mid-sentence ("Manufactu" + "rer", "...- 271" +
#         "00 Pavia - Italy"). This was the single biggest cause of rows
#         with no manufacturer/country at all, or truncated ones.
#         Fix: validate candidate splits at the row-SEGMENT level (words
#         grouped by y, each row split into x-contiguous segments) instead
#         of the per-word check -- only call it multi-column when both
#         sides have a healthy, comparable number of clean segments.
#
#  BUG 6: extract_all_from_block bailed out (returned []) after just 2
#         consecutive unparseable lines with zero records collected —
#         which is exactly what "paragraph"-style multi-manufacturer
#         blocks (name / street / city+country, separated by a blank
#         line or a standalone "Or") look like line-by-line. This caused
#         multiple manufacturers to be merged into one, with the 2nd+
#         manufacturer's name/address stuck inside the 1st's address
#         field (e.g. Suprax/J01DD08: 'Famar Lyon' + 'S.C. Zentiva S.A.'
#         merged into one row).
#         Fix: manufacturers are now split into paragraphs FIRST (on a
#         blank line or standalone "Or"/"OR" line), each parsed on its
#         own, so one manufacturer's failure/success can never swallow
#         or drop another.

import re
import logging
from collections import defaultdict

import pdfplumber

log = logging.getLogger(__name__)

# ── Compiled patterns ─────────────────────────────────────────────────────────
# BUG FIX (v7): was literal "...holder and manufacturer" only -- missed
# the common "Marketing authorisation holder/Manufacturer" (slash, no
# spaces) variant seen on e.g. Calvepen. Accept "and", "/", or "&".
_MAH_AND_MFR = re.compile(r"marketing\s+authorisation\s+holder\s*(?:and|/|&)\s*manufacturer", re.I)
# BUG FIX (v7): added "national rules" / "product" authorisation holder
# as recognised synonyms for "marketing authorisation holder" -- some
# leaflets (esp. homeopathic / traditional-registration products, and
# some OTC brands) use these instead.
_MAH_ONLY    = re.compile(r"^[\s\*]*(?:marketing|national\s+rules|product)\s+authorisation\s+holder\s*[:\*]?\s*$", re.I)
# BUG FIX (v7): was `manufacturer(?:s|\(s\))?` -- required "(s)" to
# immediately follow "manufacturer" with NO space, so "Manufacturer (s)"
# (space before the parenthetical, seen on e.g. Zofran Zydis) never
# matched. Added \s* before the suffix.
_MFR_ONLY    = re.compile(r"^[\s\*]*manufacturer\s*(?:s|\(s\))?[\u00b9\u00b2\u00b3\d]?\s*[:\*]?\s*$", re.I)
_MFR_BY      = re.compile(
    r"manufactured\s+by\s*:?|"
    r"manufactured\s+at\s*:?|"
    r"manufacturer\s+responsible\s+for\s+batch\s+release\s*:?|"
    r"alternative\s+manufacturer",
    re.I,
)
# BUG FIX (v7): was `manufacturer` (singular only, colon must follow
# immediately) -- missed "Manufacturers:" (plural, e.g. MEFAC) and
# "Manufacturer (EU QP Release Site):" (parenthetical qualifier before
# the colon, e.g. Olbas Oil / Tiger Balm). Added `s?` and an optional
# `(...)` group before the colon.
_MFR_PFX = re.compile(r"^(?:manufactured?\s+by|manufacturers?)\s*(?:\([^)]*\))?\s*:\s*", re.I)

_SKIP_LINE = re.compile(
    r"(marketing\s+authorisation\s+holder|this\s+med(icine|icinal)|further\s+info|"
    r"revision|date\s+of|contents\s+of|other\s+info|where\s+to|"
    r"last\s+revised?|not\s+applicable|to\s+be\s+completed|"
    r"names?:\s*$|^names?\s*$|^state\s*$|"
    r"^page\s+\d+\s+of\s+\d+$|^\d+\s*/\s*\d+$)",
    re.IGNORECASE,
)
_STRIP_TAIL = re.compile(r",?\s*\b(tel|fax|phone|e-?mail|www\.|http)[:\s].*$", re.I)
_EEA_NOISE  = re.compile(
    r",?\s*(?:austria|belgium|bulgaria|croatia|cyprus|czech republic|denmark|"
    r"estonia|finland|france|germany|greece|hungary|iceland|ireland|italy|"
    r"latvia|lithuania|luxembourg|malta|netherlands|norway|poland|portugal|"
    r"romania|slovakia|slovenia|spain|sweden|united kingdom)"
    r"(?:\s*[:\-]\s*[^\n,]{0,80})?"
    r"(?:,\s*(?:austria|belgium|bulgaria|croatia|cyprus|czech republic|denmark|"
    r"estonia|finland|france|germany|greece|hungary|iceland|ireland|italy|"
    r"latvia|lithuania|luxembourg|malta|netherlands|norway|poland|portugal|"
    r"romania|slovakia|slovenia|spain|sweden|united kingdom)"
    r"(?:\s*[:\-]\s*[^\n,]{0,80})?){2,}.*$",
    re.IGNORECASE | re.DOTALL,
)

_COUNTRY_MAP = [
    (r"\bireland\b", "Ireland"),
    (r"\bco\.\s*(cork|tipperary|wicklow|kildare|galway|limerick|waterford|mayo|louth|meath|sligo|wexford|kerry|clare|donegal|roscommon|longford|offaly|laois|kilkenny|carlow|cavan|monaghan|leitrim|westmeath)\b", "Ireland"),
    (r"\b(dublin|cork|galway|limerick|waterford|drogheda|dundalk|sligo|mullingar|athlone|kilkenny|wexford|tralee|carlow|naas|ennis|tullamore|clonmel|portlaoise|bray|swords|leixlip|celbridge|lucan|tallaght|blanchardstown|mulhuddart|citywest|little island|baldoyle|sandyford|bantry|blackrock)\b", "Ireland"),
    (r"\bgermany\b|\bdeutschland\b|\b[dD]-\s*\d{5}\b|\bD\s+\d{5}\b", "Germany"),
    (r"\b(munich|münchen|berlin|hamburg|frankfurt|cologne|köln|düsseldorf|stuttgart|dortmund|essen|bremen|leipzig|dresden|hannover|nuremberg|melsungen|barleben|bad vilbel|leverkusen|ingelheim|marburg|konstanz|penzberg|wiesbaden|bonn|mannheim|heidelberg|lich|ulm|regensburg|augsburg|freiburg|münster|bielefeld|wuppertal|bochum|aachen|halle|magdeburg|erfurt|holzkirchen)\b", "Germany"),
    (r"\bfrance\b|\bfran[cç]ais\b|\bF-\s*\d{5}\b", "France"),
    (r"\b(paris|lyon|gentilly|tours|amboise|chartres|strasbourg|bordeaux|marseille|toulouse|nantes|montpellier|lille|nice|rennes|reims|colombes|rueil|val.de.reuil|marcy.l.etoile|evreux|elancourt|chambray|vendôme|amilly)\b", "France"),
    (r"\bspain\b|\bespa[nñ][ah]a?\b", "Spain"),
    (r"\b(madrid|barcelona|zaragoza|seville|sevilla|valencia|bilbao|malaga|murcia|toledo|burgos|granada|alcala|alcalà|guadalajara|utebo|aranda de duero|azuqueca|miralcampo|parets del vallès|cornellà)\b", "Spain"),
    (r"\bital[yi]\b|\bitalien\b", "Italy"),
    (r"\b(rome|roma|milan|milano|naples|napoli|turin|torino|bologna|florence|firenze|genoa|genova|palermo|bari|catania|venice|venezia|verona|padova|parma|baranzate|pomezia|anagni|torre annunziata|aprilia|segrate|assago|scandicci|cinisello|liscate|latina|borgo s\. michele)\b", "Italy"),
    (r"\bpolska?\b|\bpoland\b", "Poland"),
    (r"\b(warsaw|warszawa|pabianice|poznan|kraków|krakow|wrocław|wroclaw|gdańsk|gdansk|łódź|lodz|katowice|szczecin|bydgoszcz|rzeszów|rzeszow|kutno|stryków|duchnice)\b", "Poland"),
    (r"\bnetherlands\b|\bthe netherlands\b|\bnederland\b|\bnl-\s*\d{4}\b", "Netherlands"),
    (r"\b(amsterdam|rotterdam|haarlem|utrecht|eindhoven|leusden|nijmegen|arnhem|breda|leiden|delft|tilburg|groningen|oss|boxmeer|zeist|naarden|almere)\b", "Netherlands"),
    (r"\bbelgium\b|\bbelgi[eë]\b|\bb-\s*\d{4}\b", "Belgium"),
    (r"\b(brussels|bruxelles|antwerp|antwerpen|ghent|gent|bruges|brugge|heist.op.den.berg|wavre|brasschaat|braine|laakdal|beerse|diegem|woluwé)\b", "Belgium"),
    (r"\baustria\b|\bösterreich\b|\ba-\s*\d{4}\b", "Austria"),
    (r"\b(vienna|wien|graz|linz|salzburg|innsbruck|klagenfurt|wels|kundl|langkampfen)\b", "Austria"),
    (r"\bswitzerland\b|\bschweiz\b|\bsuisse\b|\bch-\s*\d{4}\b", "Switzerland"),
    (r"\b(zurich|zürich|basel|bern|geneva|genève|lausanne|zug|allschwil|kaiseraugst|rotkreuz|stein am rhein)\b", "Switzerland"),
    (r"\bsweden\b|\bsverige\b|\bse-\s*\d{3}\s*\d{2}\b", "Sweden"),
    (r"\b(stockholm|gothenburg|göteborg|malmö|malmo|södertälje|sodertalje|uppsala|helsingborg|orebro|linköping|solna|mölndal)\b", "Sweden"),
    (r"\bdenmark\b|\bdanmark\b|\bdk-\s*\d{4}\b", "Denmark"),
    (r"\b(copenhagen|københavn|aarhus|odense|ballerup|smørum|bagsværd|bagsvard|kalundborg|hillerød|roskilde|helsingør|frederiksberg|gladsaxe|lyngby|allerød|birkerød|hørsholm|virum|charlottenlund|søborg|herlev|brøndby|albertslund|taastrup|greve|køge|næstved|slagelse|esbjerg|vejle|horsens|kolding|silkeborg|herning|viborg|randers|aalborg|fredericia)\b", "Denmark"),
    (r"\bnorway\b|\bnorge\b|\bno-\s*\d{4}\b", "Norway"),
    (r"\b(oslo|bergen|trondheim|lillestrøm)\b", "Norway"),
    (r"\bfinland\b|\bsuomi\b|\bfi-\s*\d{5}\b", "Finland"),
    (r"\b(helsinki|espoo|tampere|turku|vantaa)\b", "Finland"),
    (r"\bczech republic\b|\bczechia\b|\bčeská\b", "Czech Republic"),
    (r"\b(prague|praha|brno|ostrava|opava)\b", "Czech Republic"),
    (r"\bslovakia\b|\bslovens(ko|ká)\b", "Slovakia"),
    (r"\b(bratislava|košice|žilina|hlohovec)\b", "Slovakia"),
    (r"\bhungary\b|\bmagyar\b", "Hungary"),
    (r"\b(budapest|debrecen|komárom|komarom|pecs|győr|miskolc|vác|tatabánya)\b", "Hungary"),
    (r"\bromania\b", "Romania"),
    (r"\b(bucharest|bucurești|bucuresti|bucharest|cluj|cluj-napoca|timisoara|timișoara|iasi|iași|constanta|constanța|turda|oradea|sibiu|brașov|brasov|galati|galați|craiova|bacau|bacău|ploiesti|ploiești|arad|pitesti|pitești|brăila|braila|splaiul|unirii|sectorul|sector\s+[1-6])\b", "Romania"),
    (r"\bslovenia\b", "Slovenia"),
    (r"\b(ljubljana|maribor|novo\s*mesto|koper|celje|mengeš)\b", "Slovenia"),
    (r"\bcroatia\b", "Croatia"),
    (r"\b(zagreb|split|rijeka|koprivnica)\b", "Croatia"),
    (r"\bportugal\b|\bpt-\s*\d{4}-\d{3}\b|\bsintra\b", "Portugal"),
    (r"\b(lisbon|lisboa|porto|amadora|loures|matosinhos|queluz|barcarena|sintra|cascais|oeiras|setúbal|setubal|braga|coimbra|aveiro|leiria|faro|funchal|almada|barreiro|montijo|alcochete|mafra|odivelas|seixal|sesimbra|palmela|arruda)\b", "Portugal"),
    (r"\bgreece\b|\bgreek\b", "Greece"),
    (r"\b(athens|athina|thessaloniki|piraeus|pallini|metamorfosi|halandri|kifisia|gerakas|paiania|koropi)\b", "Greece"),
    (r"\bbulgaria\b", "Bulgaria"),
    (r"\b(sofia|plovdiv|varna|dupnitsa)\b", "Bulgaria"),
    (r"\bluxembourg\b|\bluxemburg\b", "Luxembourg"),
    (r"\b(luxembourg city|strassen|leudelange|bertrange|munsbach)\b", "Luxembourg"),
    (r"\bmalta\b|\bmsida\b|\bbirkirkara\b|\bqormi\b", "Malta"),
    (r"\bcyprus\b|\bnicosia\b|\blimassol\b|\blarnaca\b", "Cyprus"),
    (r"\bestonia\b|\btallinn\b|\btartu\b", "Estonia"),
    (r"\blatvia\b|\briga\b|\bvalmiera\b", "Latvia"),
    (r"\blithuania\b|\bvilnius\b|\bkaunas\b", "Lithuania"),
    (r"\bunited kingdom\b|\bu\.k\.\b|\buk\b|\bengland\b|\bwales\b|\bscotland\b|\bnorthern ireland\b", "United Kingdom"),
    (r"\b(london|manchester|edinburgh|glasgow|bristol|leeds|sheffield|worthing|barnard castle|loughborough|wrexham|chester|runcorn|sandwich|brentford|uxbridge|harmondsworth|weybridge|middlesex|berkshire|hampshire|kent|stockport|banbury|oxfordshire|hove|brighton|crawley|east sussex|west sussex|surrey|essex|hertfordshire|hertford|cambridge|norwich|reading|slough|swindon|nottingham|coventry|leicester|wolverhampton|stoke|derby|newcastle|sunderland|sheffield|hull|york|bath|exeter|truro|plymouth|swansea|cardiff|newport|inverness|aberdeen|dundee|belfast|derry|lisburn|armagh|newry|enniskillen|coleraine|antrim|portadown|ballymena|newtownabbey|carrickfergus|castlereagh|north down|east antrim)\b", "United Kingdom"),
    (r"\bisrael\b|\bpetah tikva\b|\bnetanya\b|\btel aviv\b|\brehovot\b|\bhaifa\b", "Israel"),
    (r"\bindia\b|\bmumbai\b|\bbangalore\b|\bhyderabad\b|\bpune\b|\bchennai\b|\bnew delhi\b", "India"),
    (r"\bchina\b|\bbeijing\b|\bshanghai\b", "China"),
    (r"\bjapan\b|\btokyo\b|\bosaka\b", "Japan"),
    (r"\bu\.s\.a\.\b|\busa\b|\bunited states\b|\bnew jersey\b|\bnew york\b", "USA"),
    (r"\bcanada\b|\btoronto\b|\bmontreal\b|\bvancouver\b", "Canada"),
    (r"\baustralia\b|\bsydney\b|\bmelbourne\b|\bbrisbane\b", "Australia"),
    (r"\bnew zealand\b|\bauckland\b", "New Zealand"),
    (r"\bsouth africa\b|\bjohannesburg\b|\bcape town\b", "South Africa"),
    (r"\bbrazil\b|\bbrasil\b", "Brazil"),
    (r"\bturkey\b|\btürkiye\b|\tankara\b|\bistanbul\b", "Turkey"),
    (r"\bserbia\b|\bbelgrade\b|\bnovi sad\b", "Serbia"),
    (r"\biceland\b|\breykjavik\b", "Iceland"),
    (r"\bsingapore\b", "Singapore"),
    (r"\bsouth korea\b|\bkorea\b|\bseoul\b", "South Korea"),
    (r"\bargentina\b|\bbuenos aires\b", "Argentina"),
    (r"\bmexico\b|\bméxico\b", "Mexico"),
    (r"\brussia\b|\bmoscow\b", "Russia"),
]
_COMPILED = [(re.compile(p, re.I), c) for p, c in _COUNTRY_MAP]


def _first_country(text):
    for pat, c in _COMPILED:
        if pat.search(text):
            return c
    return ""


def _country_from_text(text):
    if not text:
        return ""
    text  = re.sub(r"\s*[–\-]\s*", ", ", text)
    clean = _EEA_NOISE.sub("", text).strip()
    segs  = [s.strip() for s in clean.split(",") if s.strip()]
    for seg in reversed(segs):
        c = _first_country(seg)
        if c:
            return c
    return _first_country(clean)


def _collect_block(lines, start_idx, max_lines=8):
    block = []
    for line in lines[start_idx+1 : start_idx+30]:
        s = _STRIP_TAIL.sub("", line.strip()).strip()
        if not s:
            if block:
                break
            continue
        if _SKIP_LINE.search(s):
            if block:
                break
            continue
        if (_MAH_ONLY.match(s) or _MFR_ONLY.match(s) or
                s.lower().startswith("this leaflet") or
                s.lower().startswith("this medicine is auth") or
                s.lower().startswith("this medicinal")):
            if block:
                break
            continue
        block.append(s.rstrip(","))
        if len(block) >= max_lines:
            break
    return block


def _parse_block(block):
    if not block:
        return {"mfr_company": "", "mfr_address": "", "mfr_country": ""}
    company = block[0].rstrip(",")
    if _SKIP_LINE.search(company):
        return {"mfr_company": "", "mfr_address": "", "mfr_country": ""}
    rest   = block[1:]
    country, cidx, country_line_remainder = "", -1, ""
    for i in range(len(rest)-1, -1, -1):
        c = _country_from_text(rest[i])
        if c:
            country, cidx = c, i
            # BUG FIX (v8): previously the whole line containing the country
            # match (e.g. "Saint Genis Laval, France") was dropped entirely
            # from the address once its country was extracted, losing real
            # address content ("Saint Genis Laval"). Keep every segment of
            # that line except the one that IS (exactly, case-insensitive)
            # the country's own name -- so "France" is dropped but "Saint
            # Genis Laval" and "Clonmel" (which merely hints at a country
            # via a city-name match, but isn't literally the country name)
            # are kept.
            segs = [s.strip() for s in rest[i].split(",") if s.strip()]
            non_country_segs = [s for s in segs if s.strip(". ").lower() != c.lower()]
            country_line_remainder = ", ".join(non_country_segs)
            break
    addr_lines = rest[:cidx] if cidx >= 0 else rest
    addr_parts = [l.rstrip(",") for l in addr_lines]
    if country_line_remainder:
        addr_parts.append(country_line_remainder)
    if not country:
        country = _country_from_text(company) or _country_from_text(", ".join(rest))
    return {
        "mfr_company": company[:200],
        "mfr_address": ", ".join(addr_parts)[:300],
        "mfr_country": country[:100],
    }


def parse_inline_record(line):
    """Parse 'Company, Address..., Country' from a single inline line.

    Key rules:
    - Country MUST be the LAST comma-separated part only.  This prevents city
      names buried mid-address from being mistaken for a country match.
    - parts[0] is never checked for country (BUG FIX from v2).
    - Lowercase company names allowed only if they contain a legal entity suffix
      (e.g. "betapharm GmbH") — otherwise rejected as address fragments.
    """
    line = line.strip().rstrip(".")
    if not line or len(line) < 10:
        return None
    parts = [p.strip() for p in line.split(",") if p.strip()]
    if len(parts) < 2:
        return None

    # Country MUST be the last part
    country = _country_from_text(parts[-1])
    if not country:
        return None
    cidx = len(parts) - 1

    company = parts[0]
    if len(company) < 5:
        return None

    # Reject obvious address-fragment company names
    _BAD_COMPANY = re.compile(
        r"^(?:road|street|avenue|drive|lane|close|court|place|way|terrace|"
        r"floor|suite|unit|building|block|wing|piso|rua|avda?\.?|calle|"
        r"stra(?:ss|[sz])e|rue|via\s|piazza)\b",
        re.I)
    if _BAD_COMPANY.match(company):
        return None

    # Allow lowercase company names only if they contain a legal entity suffix
    if company[0].islower():
        if not _is_company_with_suffix(line):
            return None

    if re.search(r"marketing.{0,20}authoris|authorisation\s+holder",
                 company, re.I):
        return None

    address = ", ".join(parts[1:cidx]) if cidx > 1 else ""
    return {"mfr_company": company, "mfr_address": address, "mfr_country": country}


def _is_company_with_suffix(text):
    """Return True if text contains a legal entity suffix."""
    _SUFFIX = re.compile(
        r"\b(gmbh|ltd\.?|llc|inc\.?|s\.a\.?|srl\.?|s\.p\.a\.?|"
        r"n\.v\.?|b\.v\.?|\bag\b|plc\.?|corp\.?|co\.|bv|nv|as\b|ab\b|"
        r"spol\.?|oy\b|kft\.?|zrt\.?|rt\.?|dac\b)\b",
        re.I)
    return bool(_SUFFIX.search(text))

# BUG FIX (v8): _country_from_text() deliberately matches on CITY/region
# names too (not just country names) so that addresses missing an explicit
# country still resolve -- e.g. "Bantry, Co Cork" -> Ireland. But that same
# leniency caused two real bugs when _parse_inline_lines joined lines
# incrementally and accepted the FIRST match it found:
#
#   1. "Dublin 18," (a city+number, mid-address) matched "Ireland" via the
#      city-name heuristic BEFORE the real trailing "Ireland" line was
#      reached -- truncating the address early and leaving the orphaned
#      "Ireland" line to be parsed as its own bogus manufacturer
#      (mfr_company="Ireland"). Seen on e.g. Doxycycline 100 mg.
#   2. The same early-acceptance could cut a wrapped record short before
#      all of its lines were joined.
#
# Fix: distinguish an EXPLICIT country match (the matched segment ends
# with the country's own name/abbreviation, e.g. "Ireland", "19400
# Greece", "BD23 2RW, UK") from a WEAK one (matched only via a city/region
# hint buried in a longer segment, e.g. "Dublin 18"). Only finalize a
# record early on an EXPLICIT match; a weak match keeps accumulating more
# lines until either an explicit match appears or the paragraph runs out
# (in which case we're forced to use whatever we have).
_EXPLICIT_COUNTRY_TAIL_RE = re.compile(
    r"(?:" + "|".join(sorted({c for _, c in _COUNTRY_MAP}, key=len, reverse=True))
    .replace(" ", r"\s+") + r"|u\.?k\.?|england|wales|scotland|northern\s+ireland|"
    r"u\.s\.a\.|usa)\.?\s*$",
    re.I,
)


def _country_confidence(text):
    """Like _country_from_text, but also reports whether the match is an
    EXPLICIT country name/abbreviation at the end of a segment (strong --
    safe to finalize on immediately) vs a WEAK city/region-based inference
    (defer, more lines may still belong to this record)."""
    if not text:
        return "", False
    segs = [s.strip() for s in text.split(",") if s.strip()]
    for seg in reversed(segs):
        c = _first_country(seg)
        if c:
            return c, bool(_EXPLICIT_COUNTRY_TAIL_RE.search(seg.strip()))
    c = _first_country(text)
    if c:
        return c, bool(_EXPLICIT_COUNTRY_TAIL_RE.search(text.strip()))
    return "", False


def _parse_inline_lines(para_lines):
    """Parse one paragraph (a single manufacturer's lines, already split
    away from any other manufacturer by a blank line / standalone "Or"
    line) into inline 'Company, Address..., Country' record(s).

    Handles both:
      - one or more full records already on their own single line
        (e.g. "Rowex Ltd, Bantry, Co. Cork, Ireland" followed immediately
        by another manufacturer's full line, with no blank separator)
      - a record wrapped across several short lines (company name alone,
        then street, then "City, Country") -- these get progressively
        joined, but only finalized once an EXPLICIT country match is seen
        (see _country_confidence) or the paragraph is exhausted, so a
        weak mid-address city hint can never prematurely cut things short.

    If nothing ever resolves via comma-joining, falls back to _parse_block
    on the accumulated lines (name = first line, country = last line that
    contains one, everything between = address) so a paragraph is never
    silently dropped.
    """
    records = []
    pending_lines = []

    def try_finalize(force):
        nonlocal pending_lines
        if not pending_lines:
            return
        joined = ", ".join(l.rstrip(",") for l in pending_lines)
        country, explicit = _country_confidence(joined)
        if not country and not force:
            return
        if country and not explicit and not force:
            return  # weak match -- wait for a stronger one or end of paragraph
        r = parse_inline_record(joined)
        if not r:
            r = _parse_block(pending_lines)
        if r and r.get("mfr_company"):
            records.append(r)
        pending_lines = []

    for i, s in enumerate(para_lines):
        pending_lines.append(s)
        try_finalize(force=(i == len(para_lines) - 1))

    return records


def extract_all_from_block(lines, start_idx):
    """Collect ALL manufacturer records following a heading.

    BUG FIX (v5): the previous version tried to parse every line as part
    of ONE running "pending" buffer across the whole block, and gave up
    entirely (returning []) after 2 consecutive failed lines with no
    records yet collected. That is exactly what happens for the common
    "paragraph" leaflet layout:

        Famar Lyon
        Avenue de General Gaulle
        Saint Genis Laval, France
        Or
        S.C. Zentiva S.A.
        50 Theodor Pallady Blvd.
        Romania

    — "Famar Lyon" then "Avenue de General Gaulle" are 2 consecutive
    failures with 0 records collected, so the old code bailed out with
    [] and control fell through to the single-manufacturer fallback,
    which glommed everything (including the second manufacturer's name
    and address) into one "address" string and kept only the last
    country found. This is exactly the Suprax/J01DD08 bug: two
    manufacturers merged into one, with "Or" and "S.C. Zentiva S.A."
    stuck inside the address field.

    Fix: manufacturers are now split into paragraphs FIRST (on a blank
    line or a standalone "Or"/"OR" line — both already used by HPRA
    leaflets as separators between alternate manufacturers), and each
    paragraph is parsed independently and can never cause a different
    paragraph to be dropped or merged.
    """
    # BUG FIX (v8): "Company contact address:" / "For further information" /
    # "Telephone:" sections that follow the real manufacturer paragraph(s)
    # (e.g. a Pfizer-style "contact Medical Information at..." blurb) were
    # not recognised as a stop boundary, so extract_all_from_block kept
    # scanning past the real manufacturer and treated fragments of the
    # contact blurb ("Unlimited Company", "Telephone: 1800 633 363.") as
    # bogus extra manufacturer records (seen on e.g. Zithromax 250 mg).
    _STOP = re.compile(
        r"(alternative\s+manufacturer|repackaged\s+by|this\s+leaflet|"
        r"this\s+med(icine|icinal)|marketing\s+authorisation\s+holder|"
        r"last\s+revised|date\s+of\s+revision|where\s+to\s+obtain|"
        r"parallel\s+product|member\s+state|procured\s+from|"
        r"company\s+contact|for\s+further\s+information|"
        r"contact\s+medical\s+information|\btelephone\b|\bfreephone\b|"
        # BUG FIX (v8b): standard accessibility disclaimer ("For information
        # in large print, tape, CD or Braille, telephone ...") was slipping
        # through -- the old pattern required "telephone:" with a colon
        # right after, but this wording has no colon before the phone
        # number wraps to the next line (seen on e.g. Sublimaze).
        r"large\s+print|in\s+braille)",
        re.I,
    )
    _OR_ONLY = re.compile(r"^\s*or\s*[:.]?\s*$", re.I)

    # ── 1. Collect raw candidate lines, stop at a real section boundary ──
    raw = []
    for line in lines[start_idx + 1 : start_idx + 60]:
        s = _STRIP_TAIL.sub("", line.strip()).strip()
        if s and _STOP.search(s):
            break
        raw.append(s)
    while raw and not raw[-1]:
        raw.pop()
    if not raw:
        return []

    # ── 2. Split into paragraphs on blank lines / standalone "Or" lines ──
    paragraphs, current = [], []
    for s in raw:
        if not s or _OR_ONLY.match(s):
            if current:
                paragraphs.append(current)
                current = []
            continue
        current.append(s)
    if current:
        paragraphs.append(current)
    if not paragraphs:
        return []

    # ── 3. Parse each paragraph independently ────────────────────────────
    records = []
    for para in paragraphs:
        para_records = _parse_inline_lines(para)
        if para_records:
            records.extend(para_records)
            continue
        # Paragraph doesn't split into inline records at all — treat it
        # as ONE manufacturer (name / address lines / country), the same
        # shape _parse_block() expects. This is what makes a genuine
        # single-manufacturer multi-line block still work correctly.
        r = _parse_block(para)
        if r["mfr_company"]:
            records.append(r)

    return records


def _join(records, key):
    return " | ".join(r[key] for r in records if r.get(key))


def _extract_page_text(page):
    """Extract text from a page, detecting and handling 1-, 2-, and 3-column layouts.

    BUG FIX (v6): the previous per-word "spanning" check almost always passes
    on an ordinary SINGLE-column page too, because most individual words are
    narrower than the candidate gap being tested -- so few words physically
    "cross" any given x position regardless of whether the page is 1- or
    2-column. That let a coincidental gap in word start-positions (e.g. a
    short line, a right-indented "Page X of Y" footer) get misread as a
    column boundary, and the page then got cropped into two pieces mid-word
    / mid-sentence (e.g. "Manufactu" + "rer", "Teofarma ... - 271" +
    "00 Pavia - Italy"), silently corrupting the very lines the extractor is
    looking for ("Manufacturer", country names, etc.). This was by far the
    single biggest cause of missing manufacturer/country data.

    Fix: validate candidate splits at the LINE-SEGMENT level, not the word
    level. Words are grouped by y-position into rows, and each row is split
    into x-contiguous segments wherever a horizontal gap appears (this keeps
    row-aligned two-column pages, e.g. Topamax, correctly split into a left
    and right segment even though both columns share the same y). A segment
    "straddles" the candidate split if its bounding box spans both sides. A
    genuine multi-column layout has many segments cleanly confined to the
    left OR right of the split, with only a few (headers/rules) straddling
    it. A single-column layout, by contrast, has most of its segments
    straddling the split (a full-width paragraph line naturally spans the
    whole content area) and few or no segments confined to one side. Only
    treat a page as multi-column when both sides have a healthy, comparable
    number of clean segments.
    """
    words = page.extract_words(x_tolerance=3, y_tolerance=3, keep_blank_chars=False)
    if len(words) < 20:
        return page.extract_text() or ""

    pw, ph = page.width, page.height
    threshold = pw * 0.03

    # Find significant gaps in x0 positions in the middle 20-80% zone
    x0s = sorted({round(w["x0"] / 4) * 4 for w in words})
    gap_candidates = []
    for i in range(len(x0s) - 1):
        if pw * 0.20 <= x0s[i] <= pw * 0.80:
            g = x0s[i+1] - x0s[i]
            if g >= threshold:
                gap_candidates.append((g, x0s[i], x0s[i+1]))
    gap_candidates.sort(reverse=True)

    if not gap_candidates:
        return page.extract_text() or ""

    # Pick up to 2 well-separated splits (supports 3-column layouts)
    splits = [(gap_candidates[0][1] + gap_candidates[0][2]) / 2]
    for gap_size, g_start, g_end in gap_candidates[1:]:
        sp = (g_start + g_end) / 2
        if gap_size >= threshold and all(abs(sp - s) > pw * 0.15 for s in splits):
            splits.append(sp)
        if len(splits) == 2:
            break
    splits.sort()

    primary_split = splits[0]

    # BUG FIX (v6): row-segment validation of the primary split, replacing
    # the old per-word crossing check.
    rows = {}
    for w in words:
        key = round(w["top"] / 3)
        rows.setdefault(key, []).append(w)

    segments = []
    for ws in rows.values():
        ws_sorted = sorted(ws, key=lambda w: w["x0"])
        seg = [ws_sorted[0]]
        for w in ws_sorted[1:]:
            prev = seg[-1]
            if w["x0"] - prev["x1"] >= threshold:
                segments.append(seg)
                seg = [w]
            else:
                seg.append(w)
        segments.append(seg)

    total_lines = len(segments)
    straddle = left_only = right_only = 0
    for seg in segments:
        xmin = min(w["x0"] for w in seg)
        xmax = max(w["x1"] for w in seg)
        if xmin < primary_split - 5 and xmax > primary_split + 5:
            straddle += 1
        elif xmax <= primary_split + 5:
            left_only += 1
        else:
            right_only += 1

    min_side_lines = max(5, total_lines * 0.15)
    is_multi_column = (
        left_only >= min_side_lines and
        right_only >= min_side_lines and
        straddle <= total_lines * 0.35
    )
    if not is_multi_column:
        return page.extract_text() or ""

    # Crop and extract each column
    boundaries = [0] + splits + [pw]
    col_texts  = []
    for i in range(len(boundaries) - 1):
        col = page.crop((boundaries[i], 0, boundaries[i+1], ph)).extract_text() or ""
        if col.strip():
            col_texts.append(col)
    return "\n".join(col_texts)


def extract_from_pdf(pdf_path):
    empty = {"mfr_company": "", "mfr_address": "", "mfr_country": ""}
    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages = pdf.pages
            for slc in [pages[max(0, len(pages)-3):], pages[:3], pages]:
                text = "\n".join(_extract_page_text(p) for p in slc)

                # BUG FIX: was r"manufacturer" — doesn't match "Manufactured by:"
                # because "Manufactured" ends in 'd' not 'r'. Changed to stem.
                if not re.search(r"manufactur|marketing authorisation", text, re.I):
                    continue

                lines = text.splitlines()

                # ── 1. "Manufacturer(s): Company, Address, Country" ──────────
                #
                # BUG FIX (v7): the value after "Manufacturer(s):" often
                # WRAPS onto the next line(s) (e.g. Benylin: "Manufacturer:
                # Delpharm Orleans, 5 avenue de Concyr, 45071" continues on
                # the next line with "Orléans, Cedex 2, France."), and can
                # list MULTIPLE manufacturers one after another (e.g. MEFAC:
                # "Manufacturers: Dales Pharmaceuticals Ltd., ... UK." then
                # "Rowa Pharmaceuticals Ltd., ... Ireland." then a 3rd on
                # the next line). Trying parse_inline_record on just the one
                # matched line silently dropped all of this. Route the
                # prefix-stripped remainder + continuation lines through
                # extract_all_from_block (the same paragraph-aware, multi-
                # manufacturer-aware logic used by the standalone-heading
                # path) instead, falling back to the old single-line
                # behaviour if that finds nothing.
                for idx, line in enumerate(lines):
                    if _MFR_PFX.match(line.strip()):
                        remainder = _MFR_PFX.sub("", line.strip())
                        synthetic = lines[:idx] + [remainder] + lines[idx + 1:]
                        records = extract_all_from_block(synthetic, idx - 1)
                        if records:
                            return {
                                "mfr_company": _join(records, "mfr_company"),
                                "mfr_address": _join(records, "mfr_address"),
                                "mfr_country": _join(records, "mfr_country"),
                            }
                        r = parse_inline_record(remainder)
                        if r and r["mfr_company"]:
                            return r

                # ── 2. "Manufactured by:" / "Manufacturer responsible..." ────
                for idx, line in enumerate(lines):
                    if _MFR_BY.search(line.strip()):
                        records = extract_all_from_block(lines, idx)
                        if records:
                            return {
                                "mfr_company": _join(records, "mfr_company"),
                                "mfr_address": _join(records, "mfr_address"),
                                "mfr_country": _join(records, "mfr_country"),
                            }
                        block = _collect_block(lines, idx)
                        r = _parse_block(block)
                        if r["mfr_company"]:
                            if not r["mfr_country"]:
                                for sl in lines[idx + 1 : idx + 25]:
                                    c = _country_from_text(sl)
                                    if c:
                                        r["mfr_country"] = c
                                        break
                            return r

                # ── 3. Standalone "Manufacturer" heading ─────────────────────
                #
                # BUG FIX (primary — Bug 1): original code had:
                #   is_mfr_head = _MFR_ONLY.match(line.strip()) or re.searatch(...)
                # re.searatch does not exist → AttributeError on every line where
                # _MFR_ONLY.match() is None (i.e. all non-Manufacturer lines).
                # The outer except caught it silently → empty returned for all PDFs
                # that use a standalone "Manufacturer" heading (~89% of the dataset).
                # Fix: removed the redundant+broken re.searatch; _MFR_ONLY.match()
                # already covers all heading variants.
                mah_idx = next(
                    (i for i, l in enumerate(lines) if _MAH_AND_MFR.search(l)), None
                )
                search_from = mah_idx or 0
                for idx, line in enumerate(lines[search_from:], search_from):
                    if not _MFR_ONLY.match(line.strip()):   # ← FIXED
                        continue

                    records = extract_all_from_block(lines, idx)
                    if records:
                        return {
                            "mfr_company": _join(records, "mfr_company"),
                            "mfr_address": _join(records, "mfr_address"),
                            "mfr_country": _join(records, "mfr_country"),
                        }

                    r = _parse_block(_collect_block(lines, idx))
                    if r["mfr_company"]:
                        if not r["mfr_country"]:
                            # PDF reflow may split the address across non-adjacent lines
                            # (e.g. "2710-0" on one line, "89 Sintra, Portugal" several
                            # lines later after an intervening "This leaflet was last
                            # revised in").  Scan ahead for any stray country fragment.
                            for sl in lines[idx + 1 : idx + 25]:
                                c = _country_from_text(sl)
                                if c:
                                    r["mfr_country"] = c
                                    break
                        return r

                # ── 4. MAH + Manufacturer combined block ─────────────────────
                if mah_idx is not None:
                    start = mah_idx
                    for j, l in enumerate(lines[mah_idx+1:mah_idx+5], mah_idx+1):
                        if _MAH_ONLY.match(l.strip()):
                            start = j
                            break
                    block = _collect_block(lines, start)
                    r = _parse_block(block)
                    if r["mfr_company"]:
                        # FIX: _parse_block treats the whole inline line as `company`
                        # when company+address+country are on one line (e.g.
                        # "Clonmel Healthcare Ltd, Waterford Road, Clonmel, Co. Tipperary, Ireland").
                        # rest=[] so its own comma-splitting never runs -- BUT
                        # _parse_block *also* has its own country fallback
                        # (_country_from_text(company)), which fires here and
                        # finds "Ireland" INSIDE that unsplit company string.
                        # That made mfr_country non-empty, which meant the
                        # "if not r['mfr_country']" guard below never ran, so
                        # the record was returned with the entire line stuffed
                        # into mfr_company and mfr_address left blank (BUG,
                        # found while pipeline-testing v7 against real PDFs).
                        # Fix: always attempt the inline-record recovery when
                        # block is a single line -- it's strictly better
                        # whenever it succeeds (it actually separates company
                        # from address), so prefer it over _parse_block's
                        # single-line fallback rather than only using it when
                        # _parse_block found nothing at all.
                        if len(block) == 1:
                            inline = parse_inline_record(block[0])
                            if inline and inline["mfr_country"]:
                                r["mfr_company"] = inline["mfr_company"]
                                r["mfr_address"] = inline["mfr_address"]
                                r["mfr_country"] = inline["mfr_country"]
                        return r

    except Exception as exc:
        log.debug("PDF failed %s: %s", pdf_path, exc)
    return empty


def process_row(row):
    pid    = str(row["product_id"])
    result = {**row.to_dict(), "mfr_company": "", "mfr_address": "", "mfr_country": "", "error": ""}
    path   = PDF_DIR / f"{pid}.pdf"
    if path.exists():
        result.update(extract_from_pdf(path))
    return result




In [97]:
# ──────────────────────────────────────────────────────────────────────────────
# Set to False to just define the extraction functions (so Cell 5b can run)
RUN_PHASE3 = True

pdfs_on_disk = {p.stem for p in PDF_DIR.glob("*.pdf")}
done_ids     = load_done(FINAL_CSV)
df1          = pd.read_csv(PHASE1_CSV).fillna("")
queue        = df1[
    df1["product_id"].astype(str).isin(pdfs_on_disk) &
    ~df1["product_id"].astype(str).isin(done_ids)
]

print(f"PDFs on disk  : {len(pdfs_on_disk):,}")
print(f"Already done  : {len(done_ids):,}")
print(f"To process    : {len(queue):,}")

if len(queue) == 0:
    print("\n✅ Phase 3 already done — all PDFs extracted. Skipping.")
elif not RUN_PHASE3:
    print(f"\nℹ️  {len(queue):,} PDFs in queue. Functions defined.")
    print("   ▶ Run Cell 5b to test extraction on 100 PDFs.")
    print("   ▶ Set RUN_PHASE3 = True above and re-run this cell to start the full run.")
else:
    batch = []
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futures = {ex.submit(process_row, row): row for _, row in queue.iterrows()}
        pbar    = tqdm(as_completed(futures), total=len(queue), desc="Phase 3", unit="pdf")
        for f in pbar:
            row = f.result()
            batch.append(row)
            pbar.set_postfix(co=row.get("mfr_company", "")[:20],
                             country=row.get("mfr_country", ""))
            if len(batch) >= BATCH_SIZE:
                append_rows(batch, FINAL_CSV, FINAL_FIELDS)
                batch.clear()
    if batch:
        append_rows(batch, FINAL_CSV, FINAL_FIELDS)

    df_final = pd.read_csv(FINAL_CSV)
    print(f"\n✅ Phase 3 complete — {len(df_final):,} rows → {FINAL_CSV}")
    print(f"   mfr_company : {(df_final.mfr_company != '').sum():,}")
    print(f"   mfr_country : {(df_final.mfr_country != '').sum():,}")


PDFs on disk  : 6,296
Already done  : 6,296
To process    : 0

✅ Phase 3 already done — all PDFs extracted. Skipping.


## Cell 5b — Quick test: run Phase 3 on 100 PDFs only

In [99]:
TEST_CSV = "hpra_test100.csv"
TEST_N   = 100

# Pick 100 PDFs: include any known-problematic ones first, then fill randomly
df1       = pd.read_csv(PHASE1_CSV).fillna("")
pdfs_disk = {p.stem for p in PDF_DIR.glob("*.pdf")}
pool      = df1[df1["product_id"].astype(str).isin(pdfs_disk)].copy()

# Prefer rows that previously had empty or wrong-looking company names
if os.path.exists(FINAL_CSV):
    prev = pd.read_csv(FINAL_CSV).fillna("")
    bad  = prev[prev["mfr_country"] == ""]["product_id"].astype(str).tolist()
    prio = pool[pool["product_id"].astype(str).isin(bad)].head(50)
    rest = pool[~pool["product_id"].astype(str).isin(bad)].sample(
               min(TEST_N - len(prio), len(pool)), random_state=42)
    sample = pd.concat([prio, rest]).head(TEST_N)
else:
    sample = pool.sample(min(TEST_N, len(pool)), random_state=42)

print(f"Testing on {len(sample)} PDFs  ({len(sample[sample['product_id'].astype(str).isin(bad if os.path.exists(FINAL_CSV) else [])])} previously missing + rest random)")

# Remove old test file so we start fresh
if os.path.exists(TEST_CSV): os.remove(TEST_CSV)

results = []
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futures = {ex.submit(process_row, row): row for _, row in sample.iterrows()}
    pbar = tqdm(as_completed(futures), total=len(sample), desc="Test run", unit="pdf")
    for f in pbar:
        row = f.result()
        results.append(row)
        pbar.set_postfix(co=row.get("mfr_company","")[:20], country=row.get("mfr_country",""))

append_rows(results, TEST_CSV, FINAL_FIELDS)
df_test = pd.read_csv(TEST_CSV).fillna("")

filled   = (df_test["mfr_country"] != "").sum()
missing  = (df_test["mfr_country"] == "").sum()
print(f"\n── Results ──────────────────────────────────")
print(f"  With country  : {filled} / {len(df_test)}  ({filled/len(df_test)*100:.1f}%)")
print(f"  Missing       : {missing}")
print(f"\n── Sample of extracted values ───────────────")
print(df_test[["product_name","mfr_company","mfr_address","mfr_country"]].to_string(max_colwidth=45))


Testing on 100 PDFs  (50 previously missing + rest random)


Test run:   0%|          | 0/100 [00:00<?, ?pdf/s]


── Results ──────────────────────────────────
  With country  : 65 / 100  (65.0%)
  Missing       : 35

── Sample of extracted values ───────────────
                                    product_name                                   mfr_company                                   mfr_address                                   mfr_country
0   Ephedrine Hydrochloride 30mg in 1ml Solut...                                                                                                                                          
1   Benylin Cough Medicine Syrup Diphenhydram...                              Delpharm Orleans   5 avenue de Concyr, 45071, Orléans, Cedex 2                                        France
2   Truoxin I.V. 200mg / 100ml solution for i...                                                                                                                                          
3                 Diclac Relief Max 2% w / w gel                                                     

## Cell 6 — Export to Excel

In [101]:
df_final = pd.read_csv(FINAL_CSV).fillna("")

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    df_final.to_excel(writer, index=False, sheet_name="Medicines")
    ws = writer.sheets["Medicines"]

    hdr_fill = PatternFill("solid", fgColor="1F4E79")
    hdr_font = Font(bold=True, color="FFFFFF", size=11)
    for cell in ws[1]:
        cell.fill = hdr_fill; cell.font = hdr_font
        cell.alignment = Alignment(horizontal="center", wrap_text=True)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    widths = {"product_id":10,"product_name":45,"licence_number":18,"licence_holder":30,
              "active_substances":30,"atc_code":35,"package_leaflet_url":10,
              "mfr_company":35,"mfr_address":45,"mfr_country":18,"error":10}
    for ci,col in enumerate(df_final.columns,1):
        ws.column_dimensions[get_column_letter(ci)].width = widths.get(col,18)

    url_col = list(df_final.columns).index("package_leaflet_url")+1
    for r in range(2,len(df_final)+2):
        cell=ws.cell(row=r,column=url_col)
        if str(cell.value).startswith("http"):
            cell.hyperlink=cell.value; cell.value="PDF"
            cell.font=Font(color="0563C1",underline="single")

    light = PatternFill("solid", fgColor="EBF3FB")
    for r in range(2,len(df_final)+2,2):
        for c in range(1,len(df_final.columns)+1):
            ws.cell(row=r,column=c).fill=light

    s = writer.book.create_sheet("Summary")
    s["A1"].value,s["B1"].value="Metric","Value"
    for c in [s["A1"],s["B1"]]: c.font=Font(bold=True)
    filled=(df_final.mfr_country!="").sum()
    stats=[("Total products",len(df_final)),
           ("With leaflet PDF",(df_final.package_leaflet_url!="").sum()),
           ("With manufacturer company",(df_final.mfr_company!="").sum()),
           ("With manufacturer address",(df_final.mfr_address!="").sum()),
           ("With manufacturer country",filled),
           ("Country coverage",f"{filled/len(df_final)*100:.1f}%"),("","")]
    for country,cnt in df_final[df_final.mfr_country!=""]["mfr_country"].value_counts().items():
        stats.append((country,int(cnt)))
    for r,(label,val) in enumerate(stats,2):
        s.cell(r,1,label); s.cell(r,2,val)
    s.column_dimensions["A"].width=30; s.column_dimensions["B"].width=15

print(f"✅ Saved → {OUTPUT_XLSX}")
print(f"   {len(df_final):,} rows  |  {filled:,} with country ({filled/len(df_final)*100:.1f}%)")

# Also save a clean final CSV
df_final.to_csv(FINAL_CSV, index=False, encoding="utf-8")
print(f"✅ Saved → {FINAL_CSV}")

df_final[["product_name","mfr_company","mfr_address","mfr_country"]].head(5)


✅ Saved → hpra_medicines.xlsx
   6,296 rows  |  5,801 with country (92.1%)
✅ Saved → hpra_final.csv


,product_name,mfr_company,mfr_address,mfr_country
0,Benylin Children’s Dry Coughs Syrup Diphenhydr...,Delpharm Orleans,"5 avenue de Concyr, 45071 Orleans, Cedex 2",France
1,Regurin 20 mg coated tablets,Madaus GmbH,"Luetticher Str. 5, 53842 Troisdorf",Germany
2,Benylin Cough Medicine Syrup Diphenhydramine h...,,,
3,Sormon 60 mg Prolonged-release Tablets,McDermott Laboratories Ltd. t/a Gerard Laborat...,"35/36 Baldoyle Industrial Estate, Grange, Road...",Ireland | Hungary
4,Xymel Comp 37.5mg / 325mg Film-Coated Tablets,Meiji Pharma Spain S.A. | Medis International ...,"Avda de Madrid, 94, 28802 Alcalá de Henares, M...",Spain | Czech Republic | Germany | France | Ir...


In [102]:
df = pd.read_csv(FINAL_CSV).fillna("")
df[["package_leaflet_url", "mfr_country",  "mfr_address"]].to_csv("testurl.csv", index=False)
print(f"✅ Saved testurl.csv — {len(df):,} rows")

✅ Saved testurl.csv — 6,296 rows


In [103]:
df = pd.read_csv(FINAL_CSV).fillna("")
empty_country = df[df["mfr_country"] == ""]
print(f"Total products        : {len(df):,}")
print(f"With country          : {(df['mfr_country'] != '').sum():,}")
print(f"Missing country       : {len(empty_country):,}  ({len(empty_country)/len(df)*100:.1f}%)")
print(f"\nSample rows missing country:")
print(empty_country[["product_id","product_name","mfr_company","mfr_address","mfr_country"]].head(20).to_string())

Total products        : 6,296
With country          : 5,801
Missing country       : 495  (7.9%)

Sample rows missing country:
     product_id                                                                                                                                                                                   product_name                                                                mfr_company                                    mfr_address mfr_country
2         16819                                                                                              Benylin Cough Medicine Syrup Diphenhydramine hydrochloride 14 mg / 5 ml Levomenthol 1.1 mg / 5 ml                                                                                                                                      
12        40384                                                                                                                                                                 Diclac Relief Ma

In [104]:
df = pd.read_csv(FINAL_CSV).fillna("")
df[["package_leaflet_url", "mfr_address", "mfr_country"]].to_csv("testurl.csv", index=False)
print(f"✅ Saved testurl.csv — {len(df):,} rows")

✅ Saved testurl.csv — 6,296 rows


## Cell 7 — Medications with missing country + PDF links

In [106]:
import os
from IPython.display import display, HTML

df = pd.read_csv(FINAL_CSV).fillna("")
missing = df[df["mfr_country"] == ""].copy()

print(f"Missing country : {len(missing):,} / {len(df):,} products ({len(missing)/len(df)*100:.1f}%)")

def _pdf_link(url, pid):
    local = PDF_DIR / f"{pid}.pdf"
    if os.path.exists(local):
        return f'<a href="{local}" target="_blank">local PDF</a>'
    if str(url).startswith("http"):
        return f'<a href="{url}" target="_blank">PDF</a>'
    return "—"

rows_html = ""
for _, row in missing.iterrows():
    link = _pdf_link(row["package_leaflet_url"], row["product_id"])
    rows_html += (
        f"<tr>"
        f"<td>{row['product_id']}</td>"
        f"<td>{row['product_name']}</td>"
        f"<td>{row['mfr_company']}</td>"
        f"<td>{row['mfr_address']}</td>"
        f"<td>{link}</td>"
        f"</tr>\n"
    )

html = f"""
<style>
  table.mc {{border-collapse:collapse; width:100%; font-size:13px;}}
  table.mc th {{background:#1F4E79; color:#fff; padding:6px 10px; text-align:left;}}
  table.mc td {{padding:5px 10px; border-bottom:1px solid #ddd; vertical-align:top;}}
  table.mc tr:nth-child(even) td {{background:#EBF3FB;}}
  table.mc a {{color:#0563C1;}}
</style>
<table class="mc">
  <thead><tr>
    <th>Product ID</th><th>Product Name</th>
    <th>Manufacturer (extracted)</th><th>Address (extracted)</th>
    <th>PDF</th>
  </tr></thead>
  <tbody>{rows_html}</tbody>
</table>
"""
display(HTML(html))


Missing country : 495 / 6,296 products (7.9%)


Product ID,Product Name,Manufacturer (extracted),Address (extracted),PDF
16819,Benylin Cough Medicine Syrup Diphenhydramine hydrochloride 14 mg / 5 ml Levomenthol 1.1 mg / 5 ml,,,local PDF
40384,Diclac Relief Max 2% w / w gel,,,local PDF
16964,Ephedrine Hydrochloride 30mg in 1ml Solution for Injection,,,local PDF
25492,Truoxin I.V. 200mg / 100ml solution for infusion,,,local PDF
25493,Truoxin I.V. 400 mg / 200 ml solution for infusion,,,local PDF
44452,Norditropin FlexPro 15 mg / 1.5 ml solution for injection in pre-filled pen,,,local PDF
44451,Norditropin FlexPro 10 mg / 1.5 ml solution for injection in pre-filled pen,,,local PDF
24614,Zovirax 200 mg / 5 mL Oral Suspension,"Aspen Bad Oldesloe GmbH, Industriestrasse 32-",,local PDF
44450,Norditropin FlexPro 5 mg / 1.5 ml solution for injection in pre-filled pen,,,local PDF
40537,Paracetamol / Ibuprofen 1000 mg / 300mg powder for oral solution in sachet,E-Pharma Trento S.P.A,"Frazione Ravina, Via Provina, 2 – Trento (TN)",local PDF


In [107]:
full_html = f"""<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>Missing Countries</title></head>
<body>{html}</body>
</html>"""

with open("missing_countries.html", "w", encoding="utf-8") as f:
    f.write(full_html)

In [125]:
# ─────────────────────────────────────────────────────────────────────────
# Filter hpra_final.csv down to rows whose ATC code(s) match a "critical"
# list, and add a clickable PDF link for manual review.
#
# Paste this into a new cell in the notebook (it reuses FINAL_CSV / PDF_DIR
# already defined in Cell 1), or run standalone after adjusting the paths.
# ─────────────────────────────────────────────────────────────────────────
import re
import pandas as pd
from pathlib import Path

FINAL_CSV    = "hpra_final.csv"      # your Phase 3 output
CRITICAL_CSV = "Data/critical.csv"        # single column: "ATC level 5"
PDF_DIR      = Path("pdfs")          # where the downloaded PDFs live
OUTPUT_CSV   = "critical_atc_review.csv"
OUTPUT_HTML  = "critical_atc_review.html"

# atc_code field looks like:
#   "J01DD Third-generation cephalosporins, J01DD08 cefixime"
# and can list several ATC-4 / ATC-5 pairs comma-separated for products
# with more than one classification. This pulls out just the ATC-5 codes
# (1 letter, 2 digits, 2 letters, 2 digits — e.g. J01DD08).
ATC5_RE = re.compile(r"\b[A-Z]\d{2}[A-Z]{2}\d{2}\b")


def extract_atc5_codes(atc_text: str) -> set:
    return set(ATC5_RE.findall(str(atc_text).upper()))


# ── Load the critical ATC code list ─────────────────────────────────────
critical_df = pd.read_csv(CRITICAL_CSV)
critical_codes = set(
    critical_df.iloc[:, 0].dropna().astype(str).str.strip().str.upper()
)
critical_codes.discard("")
print(f"Critical ATC codes loaded: {len(critical_codes):,}")

# ── Load the scraped product data and match ─────────────────────────────
df = pd.read_csv(FINAL_CSV).fillna("")
df["_atc5_codes"] = df["atc_code"].apply(extract_atc5_codes)
df["matched_critical_atc"] = df["_atc5_codes"].apply(
    lambda codes: ", ".join(sorted(codes & critical_codes))
)
critical_rows = df[df["matched_critical_atc"] != ""].copy()
print(f"Products matching a critical ATC code: {len(critical_rows):,} / {len(df):,}")

# ── Add a local-PDF-first link (falls back to the HPRA URL) ─────────────
def pdf_link(row):
    local = PDF_DIR / f"{row['product_id']}.pdf"
    if local.exists():
        return str(local)
    url = str(row.get("package_leaflet_url", ""))
    return url if url.startswith("http") else ""

critical_rows["pdf_link"] = critical_rows.apply(pdf_link, axis=1)

# ── Save a clean CSV for review ──────────────────────────────────────────
review_cols = [
    "product_id", "product_name", "licence_number", "licence_holder",
    "atc_code", "matched_critical_atc",
    "mfr_company", "mfr_address", "mfr_country",
    "pdf_link",
]
review = critical_rows[review_cols]
review.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved {OUTPUT_CSV} — {len(review):,} rows")

# ── Also save a clickable HTML table (easier for manual checking) ───────
def _link_html(path_or_url):
    if not path_or_url:
        return "—"
    return f'<a href="{path_or_url}" target="_blank">PDF</a>'

rows_html = "\n".join(
    "<tr>"
    f"<td>{r.product_id}</td>"
    f"<td>{r.product_name}</td>"
    f"<td>{r.matched_critical_atc}</td>"
    f"<td>{r.mfr_company}</td>"
    f"<td>{r.mfr_address}</td>"
    f"<td>{r.mfr_country}</td>"
    f"<td>{_link_html(r.pdf_link)}</td>"
    "</tr>"
    for r in review.itertuples()
)

html = f"""
<style>
  table.mc {{border-collapse:collapse; width:100%; font-size:13px;}}
  table.mc th {{background:#1F4E79; color:#fff; padding:6px 10px; text-align:left;}}
  table.mc td {{padding:5px 10px; border-bottom:1px solid #ddd; vertical-align:top;}}
  table.mc tr:nth-child(even) td {{background:#EBF3FB;}}
  table.mc a {{color:#0563C1;}}
</style>
<table class="mc">
  <thead><tr>
    <th>Product ID</th><th>Product Name</th><th>Matched ATC</th>
    <th>Manufacturer</th><th>Address</th><th>Country</th><th>PDF</th>
  </tr></thead>
  <tbody>{rows_html}</tbody>
</table>
"""
with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)
print(f"✅ Saved {OUTPUT_HTML} — open it in a browser to click through PDFs")

# If running inside the notebook, you can also just display it inline:
# from IPython.display import display, HTML
# display(HTML(html))

Critical ATC codes loaded: 299
Products matching a critical ATC code: 1,165 / 6,296
✅ Saved critical_atc_review.csv — 1,165 rows
✅ Saved critical_atc_review.html — open it in a browser to click through PDFs


In [127]:
# ─────────────────────────────────────────────────────────────────────────
# Copy the PDFs for critical-ATC-code products into one folder + zip it,
# so you can send it over for checking.
#
# Paste into a new cell in the notebook (reuses FINAL_CSV / PDF_DIR from
# Cell 1). Run the filter_critical_atc.py cell first if you want the CSV/
# HTML review files too — this cell is self-contained otherwise.
# ─────────────────────────────────────────────────────────────────────────
import re
import shutil
import zipfile
import pandas as pd
from pathlib import Path

FINAL_CSV    = "hpra_final.csv"
CRITICAL_CSV = "Data/critical.csv"
PDF_DIR      = Path("pdfs")
OUT_DIR      = Path("critical_atc_pdfs")
ZIP_PATH     = "critical_atc_pdfs.zip"

ATC5_RE = re.compile(r"\b[A-Z]\d{2}[A-Z]{2}\d{2}\b")

def extract_atc5_codes(atc_text: str) -> set:
    return set(ATC5_RE.findall(str(atc_text).upper()))

# ── Load critical ATC codes ────────────────────────────────────────────
critical_df = pd.read_csv(CRITICAL_CSV)
critical_codes = set(critical_df.iloc[:, 0].dropna().astype(str).str.strip().str.upper())
critical_codes.discard("")

# ── Filter products ─────────────────────────────────────────────────────
df = pd.read_csv(FINAL_CSV).fillna("")
df["_atc5_codes"] = df["atc_code"].apply(extract_atc5_codes)
df["matched_critical_atc"] = df["_atc5_codes"].apply(
    lambda codes: ", ".join(sorted(codes & critical_codes))
)
critical_rows = df[df["matched_critical_atc"] != ""].copy()
print(f"Products matching a critical ATC code: {len(critical_rows):,} / {len(df):,}")

# ── Copy matching PDFs into one folder ──────────────────────────────────
OUT_DIR.mkdir(exist_ok=True)

copied, skipped = [], []
for _, row in critical_rows.iterrows():
    pid = str(row["product_id"])
    src = PDF_DIR / f"{pid}.pdf"
    if not src.exists():
        skipped.append(pid)
        continue
    safe_name = re.sub(r"[^\w\-. ]", "_", str(row["product_name"]))[:80]
    atc = row["matched_critical_atc"].replace(", ", "-")
    dst = OUT_DIR / f"{pid}__{atc}__{safe_name}.pdf"
    shutil.copy2(src, dst)
    copied.append(pid)

# ── Zip it up ────────────────────────────────────────────────────────────
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUT_DIR.glob("*.pdf"):
        zf.write(f, arcname=f.name)

print(f"✅ Copied {len(copied):,} PDFs → {OUT_DIR}/")
print(f"✅ Zipped → {ZIP_PATH}")
if skipped:
    print(f"⚠️  {len(skipped):,} rows had no PDF on disk (not downloaded): {skipped[:10]}{'...' if len(skipped) > 10 else ''}")

Products matching a critical ATC code: 1,165 / 6,296
✅ Copied 1,165 PDFs → critical_atc_pdfs/
✅ Zipped → critical_atc_pdfs.zip


In [129]:
# ─────────────────────────────────────────────────────────────────────────
# Filter hpra_final.csv down to rows whose ATC code(s) match a "critical"
# list, and export a CSV for review with the HPRA web URL to each PDF
# (not a local file path — so links work for anyone, anywhere).
#
# Paste this into a new cell in the notebook (reuses FINAL_CSV from Cell 1),
# or run standalone after adjusting the paths.
# ─────────────────────────────────────────────────────────────────────────
import re
import pandas as pd

FINAL_CSV    = "hpra_final.csv"      # your Phase 3 output
CRITICAL_CSV = "Data/critical.csv"        # single column: "ATC level 5"
OUTPUT_CSV   = "critical_atc_review.csv"

# atc_code field looks like:
#   "J01DD Third-generation cephalosporins, J01DD08 cefixime"
# Pulls out just the ATC-5 codes (e.g. J01DD08).
ATC5_RE = re.compile(r"\b[A-Z]\d{2}[A-Z]{2}\d{2}\b")


def extract_atc5_codes(atc_text: str) -> set:
    return set(ATC5_RE.findall(str(atc_text).upper()))


# ── Load the critical ATC code list ─────────────────────────────────────
critical_df = pd.read_csv(CRITICAL_CSV)
critical_codes = set(
    critical_df.iloc[:, 0].dropna().astype(str).str.strip().str.upper()
)
critical_codes.discard("")
print(f"Critical ATC codes loaded: {len(critical_codes):,}")

# ── Load the scraped product data and match ─────────────────────────────
df = pd.read_csv(FINAL_CSV).fillna("")
df["_atc5_codes"] = df["atc_code"].apply(extract_atc5_codes)
df["matched_critical_atc"] = df["_atc5_codes"].apply(
    lambda codes: ", ".join(sorted(codes & critical_codes))
)
critical_rows = df[df["matched_critical_atc"] != ""].copy()
print(f"Products matching a critical ATC code: {len(critical_rows):,} / {len(df):,}")

# ── Save CSV — pdf_url is the HPRA web link, not a local path ───────────
review_cols = [
    "product_id", "product_name", "licence_number", "licence_holder",
    "atc_code", "matched_critical_atc",
    "mfr_company", "mfr_address", "mfr_country",
    "package_leaflet_url",
]
review = critical_rows[review_cols].rename(columns={"package_leaflet_url": "pdf_url"})
review.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved {OUTPUT_CSV} — {len(review):,} rows")

Critical ATC codes loaded: 299
Products matching a critical ATC code: 1,165 / 6,296
✅ Saved critical_atc_review.csv — 1,165 rows
